# **Build a Mini-BERT Model From Scratch**

In [1]:
#Import Libraries

import torch
import torch.nn as nn 
from tqdm import tqdm
from transformers import BertTokenizer
import requests
from zipfile import ZipFile
from torch.utils.data import Dataset
import pandas as pd
import json
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tensorflow import Tensor
import math
import ast
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import matplotlib.pyplot as plt


C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(torch.__version__)

2.11.0+cpu


In [3]:
#define tokenizor
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [4]:
# def download (url, filename):
#     response = requests.get(url)
#     if response.status_code == 200:
#         with open(filename, 'wb') as f:
#             f.write(response.content)
#     else:
#         print(f"Failed to download. Status code: {response.status_code}")

In [5]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/bZaoQD52DcMpE7-kxwAG8A.zip'
filename = 'BERT-dataset'

In [6]:
# download(url, filename)

In [7]:
# def unzip_file (filename,directory):
#     with ZipFile(filename, 'r') as unzip_f:
#         unzip_f.extractall(directory)

In [8]:
# directory = 'D:\Projects\mini-bert-nsp-mlm-pretraining'

In [9]:
# unzip_file(filename,directory)

In [10]:
data = pd.read_csv('bert_dataset/bert_train_data.csv')
data.iloc[2]

Original Text    I rented I AM CURIOUS-YELLOW from my video sto...
BERT Input       [1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...
BERT Label       [0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...
Segment Label    1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...
Is Next                                                          1
Name: 2, dtype: object

In [11]:
row =  data.iloc[2]
ans = torch.tensor(ast.literal_eval(row["BERT Input"]))
print(ans.type)

<built-in method type of Tensor object at 0x000001CF33CE9540>


In [12]:
data.head()

,Original Text,BERT Input,BERT Label,Segment Label,Is Next
0,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 16, 94, 12615, 5, 1026, 22, 5, 201, 18, 11...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",0
1,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 16, 1571, 16, 249, 35471, 3, 67, 3, 1138, ...","[0, 0, 0, 0, 0, 0, 46, 0, 401, 0, 0, 0, 0, 5, ...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1
2,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...","[0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1
3,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 9266, 54, 14, 3, 783, 11, 2483, 17, 3, 7, ...","[0, 0, 0, 0, 134, 0, 0, 0, 0, 685, 7, 0, 0, 9,...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",0
4,I rented I AM CURIOUS-YELLOW from my video sto...,"[1, 66, 14519, 4444, 7, 4637, 76, 1479, 11, 60...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 60, 0, 0, 7552, 0,...","1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...",1


### **Build Dataset Class**

In [13]:
class BertDatatsetCSV(Dataset):
    def __init__(self, filename):

        self.dataset = pd.read_csv(filename)
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    def __len__(self):
        return len(self.dataset)

    def safe_json(self, x):
        if isinstance(x, str):
            return json.loads(x)
        return x

    def __getitem__(self, idx):
        self.dataset = self.dataset.reset_index(drop=True)
        row = self.dataset.iloc[idx]

        bert_input = torch.tensor(
            ast.literal_eval(str(row["BERT Input"]).strip()), dtype=torch.long
        )

        bert_label = torch.tensor(
            ast.literal_eval(str(row["BERT Label"]).strip()), dtype=torch.long
        )

        segment_label = torch.tensor(
            ast.literal_eval(str(row["Segment Label"]).strip()), dtype=torch.long
        )

        is_next = torch.tensor(row["Is Next"], dtype=torch.long)

        return bert_input, bert_label, segment_label, is_next

In [14]:
row =  data.iloc[31360]
bert_input = torch.tensor(json.loads(row['BERT Input']), dtype=torch.long)
bert_input

tensor([   1,   49,   12,   19,   47,   84,  365,  135,    6,    2,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,   21, 1040,
           7,    5,    3,    3,   94,  135,    3, 1142,    3,   22,  294, 8429,
           7,    3,  143,    3,   34,   32,    3,    2])

In [15]:
data = data.reset_index(drop=True)
row = data.iloc[3]

bert_input = torch.tensor(
        ast.literal_eval(str(row["BERT Input"]).strip()), dtype=torch.long
    )
bert_input

tensor([    1,  9266,    54,    14,     3,   783,    11,  2483,    17,     3,
            7,  1578,   121,     3,   345,    10,   117,  1163,     3,    16,
           75,    78,     3,    77,     3,    22,   540,     6,     2,     0,
           15,   206,  2185,  7274,     3,  1922,     3,    10, 21481,    53,
            3,  4659,    31,  2384,     7,    64,    55,   405,    23,    50,
          477,  1695,     7,  8138,     7,     8,  1002,     3,     6,     2])

### **Build Collate Function**

In [16]:
PAD_IDX = 0

def collate_func(batch):
    bert_input_batch = []
    bert_label_batch = []
    segment_label_batch = []
    is_next_batch = []

    for bert_input, bert_label, segment_label, is_next in batch:
        bert_input_batch.append(bert_input)
        bert_label_batch.append(bert_label)
        segment_label_batch.append(segment_label)
        is_next_batch.append(is_next)

    bert_input_batch = pad_sequence(
        bert_input_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    bert_label_batch = pad_sequence(
        bert_label_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    segment_label_batch = pad_sequence(
        segment_label_batch,
        batch_first=False,
        padding_value=PAD_IDX
    )

    is_next_batch = torch.tensor(is_next_batch, dtype=torch.long)

    return bert_input_batch, bert_label_batch, segment_label_batch, is_next_batch

### **Initialize the Dataset and Dataloaders**

In [17]:
train_dataset = BertDatatsetCSV("./bert_dataset/bert_train_data.csv")
test_dataset = BertDatatsetCSV("./bert_dataset/bert_test_data.csv")

In [18]:
train_dataset.__len__()

170540

In [19]:
test_dataset.__len__()

167449

In [20]:
batch_size = 4

In [21]:
train_dataloader = DataLoader(train_dataset,batch_size=batch_size, collate_fn=collate_func,shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size, collate_fn=collate_func)

### **Model Implementation**

Model Creation (BERT Embeddings)

In BERT, three types of embeddings are combined to represent each input token:

1. Token Embedding

This is the basic representation of each word/token.

* Maps every token to a fixed-size dense vector
* Learns semantic meaning of words
* Captures relationships between tokens based on context

👉 Think of it as: “What does this word mean?”

2. Positional Embedding

Transformers process tokens in parallel, so they don’t naturally understand order.

* Adds position information to each token
* Encodes where a token appears in the sequence
* Helps the model understand word order and structure

👉 Think of it as: “Where is this word in the sentence?”

3. Segment Embedding

Used when BERT processes multiple sentences (e.g., sentence pairs).

* Assigns different embeddings to different segments (Sentence A, Sentence B)
* Helps distinguish between parts of the input
* Useful for tasks like question answering and next sentence prediction

👉 Think of it as: “Which sentence does this word belong to?”

Final Input Representation

The final embedding for each token is the sum of all three:

* Final Embedding = Token Embedding + Positional Embedding + Segment Embedding

This combined representation allows BERT to understand meaning, order, and context simultaneously.

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

#### **Token Embeddings**

In [23]:
class TokenEmbeddings (nn.Module):
    
    def __init__(self, vocab_size, num_dims):
        super().__init__()
        self.num_dims = num_dims
        self.embeddings = nn.Embedding(vocab_size, num_dims)
    
    def forward(self, input_tokens:Tensor):
        embedding_out = self.embeddings(input_tokens.long())
        return embedding_out * math.sqrt(self.num_dims)

#### **Position Embedding**

In [24]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len, d_model, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        positions = torch.arange(0, max_seq_len).unsqueeze(1)

        pe = torch.zeros((max_seq_len, d_model))

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)

        pe = pe.unsqueeze(1)
        self.register_buffer("positional_encodings", pe)

    def forward(self, word_embeddings):
        # word_embeddings -> [seq_len, batch_size, num_dims]
        seq_len = word_embeddings.size(0)
        positional_embeddings = (
            word_embeddings + self.positional_encodings[0:seq_len, :, :]
        )
        return self.dropout(positional_embeddings)

#### **BERT Embeddings**

In [25]:
class BERTEmbeddings (nn.Module):
    
    def __init__(self, vocab_size, d_models,max_seq_len, dropout):
        super().__init__()
        self.token_embeddings = TokenEmbeddings(vocab_size, d_models)
        self.positional_embeddings = PositionalEmbeddings(max_seq_len,d_models,dropout)
        self.segment_embeddings = nn.Embedding(3,d_models)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self,word_tokens, segment_ids):
        embed_out = self.token_embeddings(word_tokens)
        pos_embed = self.positional_embeddings(embed_out)
        segment_embed = self.segment_embeddings(segment_ids)
        
        bert_embeddings = pos_embed + segment_embed
        return self.dropout(bert_embeddings)

#### **Initiate Sample Dataset for Testing**

In [26]:
count = 0

for batch in train_dataloader:
    bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
    
    count += 1
    if count == 5:
        break

### **BERT Model Architecture Overview**

* The complete BERT model consists of the following key components:

1. Initialization

The BERT class extends torch.nn.Module. During initialization, it defines core hyperparameters such as:

* Vocabulary size
* Hidden/model dimension
* Number of encoder layers
* Number of attention heads
* Dropout rate

These parameters configure the overall architecture of the model.

2. Embedding Layer

The embedding layer is implemented using the BERTEmbedding class. It combines:

* Token embeddings
* Segment embeddings

This layer converts input tokens into dense vector representations suitable for the transformer.

3. Transformer Encoder

A stack of Transformer Encoder layers processes the embeddings. Each layer applies:

* Multi-head self-attention
* Feedforward neural networks

The depth (number of layers), number of attention heads, model dimension, and dropout are controlled by the initialized parameters.

4. Next Sentence Prediction (NSP) Head

A linear classification layer is used for the NSP task. It:

* Takes the encoder output (typically the [CLS] token representation)
* Predicts whether two sentences are consecutive
* Outputs probabilities for two classes (IsNext / NotNext)

5. Masked Language Modeling (MLM) Head

Another linear layer handles the MLM task. It:

* Uses encoder outputs for all token positions
* Predicts the original tokens for masked positions
* Produces probability distributions over the vocabulary

6. Forward Pass

The forward method defines how data flows through the model:

* Inputs: bert_inputs, segment_labels
* Process: Embedding → Transformer Encoder → Task-specific heads

Outputs:
* NSP predictions
* MLM predictions

In [27]:
class BERTModel (nn.Module):
    
    def __init__(self, vocab_size, d_model, max_seq_len, n_layers, n_heads, dropout = 0.1):
        
        super().__init__()
        #define bert embedding
        self.bert_embedding = BERTEmbeddings(vocab_size,d_model,max_seq_len,dropout)
        # define encoder layer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads, 
                                                        dropout=dropout, batch_first=False)
        # define encoder
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,num_layers=n_layers)
        
        #define linear layer for mask language modeling
        self.fc_mlm = nn.Linear(d_model, vocab_size)
        
        #define linear layer for next word prediction
        self.fc_nsp = nn.Linear(d_model, 2)
        
    def forward(self,bert_inputs, segment_labels):
        bert_out = self.bert_embedding(bert_inputs, segment_labels)
        
        #we need to create padding mask to prevent attention to padding
        pad_mask = (bert_inputs == PAD_IDX).transpose(0,1)
        
        encoder_out = self.encoder(bert_out, src_key_padding_mask = pad_mask)
        
        # for mlm
        mlm_out = self.fc_mlm(encoder_out)
        
        #for nsp
        nsp_out = self.fc_nsp(encoder_out[0,:,:])
        
        return mlm_out, nsp_out

#### **Create a Instance of the Model**

In [28]:
vocab_size = 147161
d_model = 12
n_layers = 2
nheads = 4
dropout = 0.1

In [29]:
bert_model = BERTModel(vocab_size,d_model,1000,n_layers,nheads,dropout)

C:\Users\Vish\AppData\Local\Temp\ipykernel_7236\3945956926.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,num_layers=n_layers)


In [30]:
padding_mask = (bert_inputs == PAD_IDX).transpose(0,1)
padding_mask.shape

torch.Size([4, 62])

In [31]:
bert_inputs.shape

torch.Size([62, 4])

In [32]:
bert_inputs[:,0]

tensor([   1, 1335,    3,    3,    3, 3200,   21,    3,    3,   22, 7100,    6,
           2,   75,    3,    6,    2,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0])

In [33]:
token_embddings = TokenEmbeddings(vocab_size,d_model)

t_embeddings = token_embddings(bert_inputs)
t_embeddings.shape


torch.Size([62, 4, 12])

In [34]:
positional_encoding = PositionalEmbeddings(max_seq_len=500,d_model=d_model,dropout=dropout)
pe = positional_encoding(t_embeddings)
pe.shape    

torch.Size([62, 4, 12])

In [35]:
for i in range(3):
    print(f"postional embddings  for the {i}th token of the first sample: {pe[i,0,:]}")

postional embddings  for the 0th token of the first sample: tensor([-0.6393,  1.7536, -0.0000,  1.3427,  2.8259, -7.3870, -6.3553,  0.2580,
        -3.3649,  5.5508, -0.4519,  7.4948], grad_fn=<SelectBackward0>)
postional embddings  for the 1th token of the first sample: tensor([-1.6372,  2.5645, -1.5052, -0.0000,  3.0864,  2.9727,  0.4749, -4.6306,
         1.2091,  0.0000, -0.3041,  4.6473], grad_fn=<SelectBackward0>)
postional embddings  for the 2th token of the first sample: tensor([ 1.7882, -1.3475,  4.5909, -2.9587,  6.4773,  4.1103, -0.6817,  4.9955,
         1.8037, -0.9158, -4.1741,  6.0422], grad_fn=<SelectBackward0>)


In [36]:
segment_embedding = nn.Embedding(3,d_model)
se = segment_embedding(segment_labels)
se.shape

torch.Size([62, 4, 12])

In [37]:
for i in range(3):
    print(f"segment embddings  for the {i}th token of the first sample: {se[i,0,:]}")

segment embddings  for the 0th token of the first sample: tensor([ 0.2380,  1.5119,  1.0049, -0.2908,  0.0891, -0.2754,  0.7668,  0.9229,
        -1.5091, -0.2285, -0.2339, -1.1568], grad_fn=<SelectBackward0>)
segment embddings  for the 1th token of the first sample: tensor([ 0.2380,  1.5119,  1.0049, -0.2908,  0.0891, -0.2754,  0.7668,  0.9229,
        -1.5091, -0.2285, -0.2339, -1.1568], grad_fn=<SelectBackward0>)
segment embddings  for the 2th token of the first sample: tensor([ 0.2380,  1.5119,  1.0049, -0.2908,  0.0891, -0.2754,  0.7668,  0.9229,
        -1.5091, -0.2285, -0.2339, -1.1568], grad_fn=<SelectBackward0>)


In [38]:
bert_embeddings = pe + se
bert_embeddings.shape

torch.Size([62, 4, 12])

In [39]:
for i in range(3):
    print(f"bert embddings  for the {i}th token of the first sample: {bert_embeddings[i,0,:]}")

bert embddings  for the 0th token of the first sample: tensor([-0.4013,  3.2655,  1.0049,  1.0519,  2.9150, -7.6624, -5.5884,  1.1809,
        -4.8740,  5.3223, -0.6858,  6.3380], grad_fn=<SelectBackward0>)
bert embddings  for the 1th token of the first sample: tensor([-1.3992,  4.0765, -0.5003, -0.2908,  3.1755,  2.6974,  1.2418, -3.7077,
        -0.3000, -0.2285, -0.5379,  3.4904], grad_fn=<SelectBackward0>)
bert embddings  for the 2th token of the first sample: tensor([ 2.0262,  0.1645,  5.5958, -3.2495,  6.5663,  3.8350,  0.0851,  5.9184,
         0.2945, -1.1442, -4.4079,  4.8854], grad_fn=<SelectBackward0>)


#### **Idea About the Encdoing Layers (Optional)**

In [40]:
encoding_layer = nn.TransformerEncoderLayer(d_model,nheads,dropout=dropout, batch_first=False)
transformer_encoder = nn.TransformerEncoder(encoder_layer=encoding_layer, num_layers=n_layers)

#pass the bert embeddings to transformer encoder
te = transformer_encoder(bert_embeddings, src_key_padding_mask = padding_mask)
te.shape

C:\Users\Vish\AppData\Local\Temp\ipykernel_7236\3737590608.py:2: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  transformer_encoder = nn.TransformerEncoder(encoder_layer=encoding_layer, num_layers=n_layers)


torch.Size([62, 4, 12])

In [41]:
for i in range(3):
    print(f"transformer encoder  for the {i}th token of the first sample: {te[i,0,:]}")

transformer encoder  for the 0th token of the first sample: tensor([ 0.1861,  0.3341, -0.7969,  0.2999,  0.5605, -1.4401, -0.8716, -0.7653,
        -0.9690,  1.5522, -0.0993,  2.0093], grad_fn=<SelectBackward0>)
transformer encoder  for the 1th token of the first sample: tensor([-1.5877,  0.1727, -0.6007, -0.3900,  0.4377,  0.4538,  2.0166, -0.9629,
        -1.3793,  0.0283,  0.8813,  0.9302], grad_fn=<SelectBackward0>)
transformer encoder  for the 2th token of the first sample: tensor([-1.0674, -0.8805,  0.7263, -1.3515,  1.2487,  0.6114,  0.6253,  1.8064,
        -0.3987, -0.7597, -1.0710,  0.5106], grad_fn=<SelectBackward0>)


##### **Next Sentence Prediction**

In [42]:
nsp_layer = nn.Linear(d_model,2)
nsp = nsp_layer(te[0,:])
nsp.shape

torch.Size([4, 2])

In [43]:

print(nsp)

tensor([[-0.6531,  0.1652],
        [-0.8585, -0.0438],
        [-0.6764,  0.0252],
        [-0.9499, -0.1896]], grad_fn=<AddmmBackward0>)


##### **Mask Language Modeling**

In [44]:
mlm_layer = nn.Linear(d_model, vocab_size)
mlm = mlm_layer(te)

mlm.shape

torch.Size([62, 4, 147161])

In [45]:
print(mlm[0:3,:,:])

tensor([[[ 0.5540, -0.6174, -0.7761,  ...,  0.1088,  0.2971,  0.5898],
         [ 0.4962, -0.6604, -0.6690,  ...,  0.2062,  0.0927,  0.6657],
         [ 0.4186, -1.1033, -0.6426,  ..., -0.1762, -0.6103,  0.1057],
         [ 0.4677,  0.1585, -0.9228,  ...,  0.1068, -0.2069,  0.3162]],

        [[-1.0292, -0.5170, -0.6431,  ..., -0.1133, -1.0555, -0.4344],
         [-0.5170,  0.9910, -0.6562,  ..., -0.1397,  0.1077,  0.0755],
         [ 0.0140, -1.0343, -0.5530,  ..., -0.4628, -1.2435, -0.3536],
         [-0.0645, -1.1165, -0.6568,  ..., -0.4302, -1.2444, -0.0023]],

        [[-0.7725,  1.3152, -0.6562,  ..., -0.0105, -0.2311, -0.0037],
         [ 0.1116, -0.9921, -0.1386,  ..., -0.8942, -0.7805, -0.0056],
         [ 0.2940, -0.5637, -0.4054,  ..., -1.0793, -1.1694, -0.7700],
         [-0.6905,  0.9972, -0.4765,  ..., -0.2591, -0.1832,  0.2330]]],
       grad_fn=<SliceBackward0>)


In [46]:
bert_model = bert_model.to(device)

## **Model Evaluation, Training and Prediction**

In [47]:
# The loss function must ignore PAD tokens and only calculates loss for the masked tokens
mlm_loss = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
nsp_loss = nn.CrossEntropyLoss()

### **Model Evaluation** 

In [48]:
def model_eval(model:BERTModel, dataloader:DataLoader, nsp_loss = nsp_loss, mlm_loss = mlm_loss, device=device):
    
    model.eval()
    
    nsp_loss_val = 0
    mlm_loss_val = 0
    total_batches = 0
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
            
            #predict the values from the model
            mlm_out_predict, nsp_out_predict = model(bert_inputs, segment_labels)
            
            # calculate loss for each prediction
            mlm_loss_batch = mlm_loss(mlm_out_predict.view(-1,mlm_out_predict.size(-1)),bert_labels.view(-1))
            
            nsp_loss_batch = nsp_loss(nsp_out_predict, is_nexts.view(-1))
            
            batch_loss = mlm_loss_batch + nsp_loss_batch
            
            if torch.isnan(batch_loss):
                continue
            else:
                total_loss += batch_loss.item()
                nsp_loss_val += nsp_loss_batch.item()
                mlm_loss_val += mlm_loss_batch.item()
                
                total_batches += 1
    
    avg_loss = total_loss / (total_batches )
    avg_nsp_loss = nsp_loss_val / (total_batches )
    avg_mlm_loss = mlm_loss_val / (total_batches)
    
    print(f"Average Loss: {avg_loss:.4f}, Average Next Sentence Loss: {avg_nsp_loss:.4f}, Average Mask Loss: {avg_mlm_loss:.4f}")
    return avg_loss

Hence our dataset is huge it may takes several hours to train the model. because of that we use sample dataset to train the model

In [49]:
train_sample_path = "./bert_dataset/bert_train_data_sampled.csv"
test_sample_path = "./bert_dataset/bert_test_data_sampled.csv"

train_sample_dataset = BertDatatsetCSV(train_sample_path)
test_sample_dataset = BertDatatsetCSV(test_sample_path)

In [50]:
train_sample_dataloader = DataLoader(train_sample_dataset,batch_size=batch_size,
                                    shuffle=True, collate_fn=collate_func)

test_sample_dataloader = DataLoader(test_sample_dataset,batch_size=batch_size, 
                                    shuffle=False, collate_fn=collate_func)


#### Average Losses Before Model Training

In [51]:
# average_loss = model_eval(bert_model,test_sample_dataloader)
# average_loss

In [52]:
# print(average_loss)

### **Model Training**


In [53]:
optimizer = AdamW(bert_model.parameters(),lr=1e-4, weight_decay=0.01, betas=(0.9,0.999))

In [54]:
num_epochs = 1
total_step = num_epochs * len(train_sample_dataloader)
warm_steps = int(total_step * 0.1)

scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps=warm_steps,
                                            num_training_steps=total_step)

In [55]:
def model_train(model:BERTModel, dataloader:DataLoader, nsp_loss=nsp_loss,
                mlm_loss = mlm_loss, optimizer = optimizer, lr_scehduler = scheduler,
                device = device):
    
    train_loss = []
    eval_loss = []
    
    for epoch in tqdm(range(num_epochs),desc="Training Epochs"):
        
        model.train()
        epoch_loss = 0
        
        for step_id, batch in enumerate(tqdm(dataloader,desc=f"epoch {epoch+1}")):
            bert_inputs, bert_labels, segment_labels, is_nexts = [b.to(device) for b in batch]
            
            #zero gradient optimizer
            optimizer.zero_grad()
            #model prediction
            mlm_predict, nsp_predict = model(bert_inputs, segment_labels)
            
            #nsp loss value defined
            nsp_loss_val = nsp_loss(nsp_predict, is_nexts.view(-1))
            #mlm loss value defined
            mlm_loss_val = mlm_loss(mlm_predict.view(-1,mlm_predict.size(-1)), bert_labels.view(-1))
            
            # total loss tensor
            loss = nsp_loss_val + mlm_loss_val
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.1)
            optimizer.step()
            scheduler.step()
            
            epoch_loss += loss.item()
    
        avg_loss_train = epoch_loss / len(dataloader)
        train_loss.append(avg_loss_train)
        print(f"Epoch {epoch+1} - Average training loss: {avg_loss_train:.4f}")
    
        avg_loss_test = model_eval(model,test_sample_dataloader)        
        eval_loss.append(avg_loss_test)
        print(f"Epoch {epoch+1} - Average testing loss: {avg_loss_test:.4f}")
    
    return avg_loss_train, avg_loss_test      
    

In [56]:
#avg_train_loss, avg_test_loss = model_train(bert_model,train_sample_dataloader)

#### **Plotting Outputs**

In [57]:
plt.figure(figsize=(6,4))
plt.scatter(range(1,num_epochs+1), avg_train_loss,label = "Training Loss", color = "blue")
plt.scatter(range(1,num_epochs+1), avg_test_loss, label = "Validation Loss", color = "red")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Train Loss VS Validation Loss")
plt.legend()
plt.show()

## noted we have only one epoch data. beacause of computational power consumption we do not go further 
## epochs.

NameError: name 'avg_train_loss' is not defined

<Figure size 600x400 with 0 Axes>

### **Model Prediction** 

##### **Next Sentence Prediction** 

In [ ]:
s1="Hi hello how are you."
s2 = "Im fine thank you."
tokens = tokenizer(s1, s2, return_tensors = 'pt')
tokens

mlm_predict, nsp_predict = bert_model(tokens['input_ids'].transpose(0,1),tokens['token_type_ids'].transpose(0,1))
nsp_predict[0]

tensor([-0.3764, -0.4342], grad_fn=<SelectBackward0>)

In [ ]:
predict = nsp_predict[0].unsqueeze(0)
prob = torch.softmax(predict,dim=1)
torch.argmax(prob,dim=1)

tensor([0])

In [ ]:
input_ids_tenosr = tokens["input_ids"]
input_ids_tenosr.shape
input_ids_tenosr.transpose(0,1)
#so we canh see our input like -> [batch_size, seq_len]
#but we want like -> [seq_len, batch_size]


torch.Size([1, 14])

In [ ]:
input_ids_tenosr_transpose = input_ids_tenosr.transpose(0,1)
input_ids_tenosr_transpose.shape

torch.Size([14, 1])

In [ ]:
def predict_nsp(sentence_1, sentence_2, model: BERTModel, tokenizer: BertTokenizer):

    model.eval()
    
    tokens = tokenizer(sentence_1, sentence_2, return_tensors="pt")
    input_ids_tenosr = tokens['input_ids'].transpose(0,1)
    label_ids_tensor = tokens['token_type_ids'].transpose(0,1)
    
    with torch.no_grad():
        mlm_predict, nsp_predict = model(input_ids_tenosr,label_ids_tensor)
        
        # we choose first logit value
        first_logit = nsp_predict
        logit = torch.softmax(first_logit,dim=1)
        predict_value = torch.argmax(logit,dim=1).item()
    
    return "Second sentence follows the first sentence" if predict_value == 1 else "Second sentence doesnot follow the first sentence"


In [ ]:
s1="Hi hello how are you."
s2 = "Im fine thank you."

print(predict_nsp(s1,s2,bert_model,tokenizer))

Second sentence doesnot follow the first sentence


#### **Mask Language Modeling Prediction**

In [110]:
input_sentence = "The cat is sat on [MASK]"
tokens = tokenizer(input_sentence, return_tensors='pt')
input_ids = tokens['input_ids'].transpose(0,1)
token_type_ids = tokens['token_type_ids'].transpose(0,1)
tokens

{'input_ids': tensor([[ 101, 1996, 4937, 2003, 2938, 2006,  103,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [111]:
tokens['input_ids'].shape

torch.Size([1, 8])

In [112]:
mlm_out, nsp_out = bert_model(input_ids,token_type_ids)
mlm_out, mlm_out.shape

(tensor([[[-0.3227,  0.3844, -0.1337,  ..., -0.3164, -0.2700, -0.0320]],
 
         [[ 0.4279, -0.7247, -0.2995,  ...,  0.2438, -1.1459,  0.0019]],
 
         [[ 0.2227, -0.0552, -1.1742,  ...,  0.6510, -0.7446, -0.4627]],
 
         ...,
 
         [[-1.2334, -0.0106, -0.9717,  ..., -0.2905, -0.1904, -0.9476]],
 
         [[ 0.7516,  0.3369, -0.4894,  ..., -1.3268,  0.2964,  0.4680]],
 
         [[ 1.1017, -0.4379, -0.0282,  ..., -1.3104, -0.1974, -0.0101]]],
        grad_fn=<ViewBackward0>),
 torch.Size([8, 1, 147161]))

In [113]:
input_sentence.split()

['The', 'cat', 'is', 'sat', 'on', '[MASK]']

In [114]:
index = 0
for idx, token in enumerate(input_sentence.split()):
    if (token == '[MASK]'):
        index = idx
        print(idx)


5


In [125]:

mask_idx_tuple = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
## (batch_index, token_position)
mask_idx = mask_idx_tuple[0]
mask_idx

tensor([6])

In [126]:
logit_predict = torch.argmax(mlm_out[mask_idx.item(),0,:], dim=-1)
logit_predict

tensor(8652)

In [127]:
predict_token = tokenizer.convert_ids_to_tokens([logit_predict.item()])[0]
predict_token

'glow'

In [128]:
def mlm_predict (sentence:str, model:BERTModel, tokenizer:BertTokenizer):
    
    model.eval()
    
    tokens = tokenizer(sentence, return_tensors='pt')
    input_ids = tokens['input_ids'].transpose(0,1)
    token_type_ids = tokens['token_type_ids'].transpose(0,1)
    
    with torch.no_grad():
        
        mlm_out, nsp_out = model(input_ids,token_type_ids)
        mask_idx = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        
        logit_predict = torch.argmax(mlm_out[mask_idx.item(),0,:], dim=-1)
        predict_token = tokenizer.convert_ids_to_tokens([logit_predict.item()])[0]
        
        predict_sentence = sentence.replace(tokenizer.mask_token,predict_token,1)
    
    return predict_sentence
        
        